In [8]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import os, sys, importlib
from pathlib import Path

KB_DIR = "/content/drive/MyDrive/AI_Trichologist/knowledgebase"
os.environ["KB_DIR"] = KB_DIR
sys.path.insert(0, str(Path(KB_DIR) / "scripts"))

import reasoning_engine as re_engine
importlib.reload(re_engine)

print("Engine:", re_engine.__file__)
print("KB:", re_engine.resolve_kb_dir())

Engine: /content/drive/MyDrive/AI_Trichologist/knowledgebase/scripts/reasoning_engine.py
KB: /content/drive/MyDrive/AI_Trichologist/knowledgebase


In [81]:
%%writefile /content/drive/MyDrive/AI_Trichologist/knowledgebase/scripts/reasoning_engine.py
import os
import json
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

WEIGHTS = {"face": 0.40, "texture": 0.25, "density": 0.20, "hair_loss": 0.15}
ENHANCEMENT_WEIGHT = 0.10
MAX_PER_CATEGORY = 2

ALLOWED_FACE = {"oval", "round", "square", "oblong", "heart"}
ALLOWED_TEXTURE = {"straight", "wavy", "curly"}
ALLOWED_DENSITY = {"low", "medium", "high"}


def resolve_kb_dir(kb_dir: Optional[str] = None) -> Path:
    if kb_dir:
        p = Path(kb_dir).expanduser().resolve()
        if p.exists():
            return p
    env = os.environ.get("KB_DIR")
    if env:
        p = Path(env).expanduser().resolve()
        if p.exists():
            return p
    default_drive = Path("/content/drive/MyDrive/AI_Trichologist/knowledgebase").resolve()
    if default_drive.exists():
        return default_drive
    return Path(__file__).resolve().parent.parent


def _load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_taxonomy_and_rules(kb_dir: Optional[str] = None) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    KB = resolve_kb_dir(kb_dir)
    taxonomy_path = KB / "data_final" / "canonical_taxonomy_final.json"
    rules_path = KB / "data_knowledge" / "hairstyle_rules.json"

    if not taxonomy_path.exists():
        raise FileNotFoundError(f"Missing taxonomy JSON: {taxonomy_path}")
    if not rules_path.exists():
        raise FileNotFoundError(f"Missing rules JSON: {rules_path}")

    taxonomy = _load_json(taxonomy_path)
    rules = _load_json(rules_path)

    if not isinstance(taxonomy, list) or not isinstance(rules, list):
        raise ValueError("taxonomy and rules must both be JSON arrays (lists).")

    return taxonomy, rules


def validate_weights(weights: Dict[str, float]) -> None:
    total = sum(weights.values())
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Weights must sum to 1.0. Current sum: {total}")


def safe_get(d: Any, key: str, default=None):
    if isinstance(d, dict):
        return d.get(key, default)
    return default


def hairloss_score_from_profile(rule: Dict[str, Any], user_profile: Dict[str, Any]) -> Tuple[float, str]:
    hl = safe_get(rule, "hair_loss", {}) or {}

    norwood = user_profile.get("norwood_stage") or user_profile.get("norwood") or user_profile.get("norwoodStage")
    norwood_score = None
    if norwood is not None:
        norwood_map = safe_get(hl, "norwood", {}) or {}
        norwood_score = norwood_map.get(str(norwood))

    pat = user_profile.get("baldness_pattern_label") or user_profile.get("baldness_pattern") or "none"
    pat = str(pat).lower().strip()

    frontal = hl.get("receding_hairline", None)
    crown = hl.get("crown_thinning", None)

    pattern_score = None
    pattern_reason = None

    if pat in ["none", "normal", "no", "0"]:
        pattern_score = None
        pattern_reason = "pattern=none"
    elif pat in ["frontal", "front", "temple", "hairline"]:
        pattern_score = frontal if frontal is not None else None
        pattern_reason = "pattern=frontal→receding_hairline"
    elif pat in ["crown", "vertex"]:
        pattern_score = crown if crown is not None else None
        pattern_reason = "pattern=crown→crown_thinning"
    elif pat in ["diffuse", "diffuse_thinning", "overall"]:
        if frontal is not None and crown is not None:
            pattern_score = (float(frontal) + float(crown)) / 2.0
            pattern_reason = "pattern=diffuse→avg(frontal,crown)"
        elif frontal is not None:
            pattern_score = float(frontal)
            pattern_reason = "pattern=diffuse→fallback frontal"
        elif crown is not None:
            pattern_score = float(crown)
            pattern_reason = "pattern=diffuse→fallback crown"
        else:
            pattern_score = None
            pattern_reason = "pattern=diffuse→no keys"
    else:
        pattern_score = None
        pattern_reason = f"pattern={pat}→ignored"

    if pattern_score is not None:
        return float(pattern_score), pattern_reason

    if norwood_score is not None:
        return float(norwood_score), f"norwood={norwood}"

    return 0.5, "neutral"


def compute_score(style: Dict[str, Any], rule: Dict[str, Any], user_profile: Dict[str, Any]) -> Tuple[float, Dict[str, Any]]:
    face_shape = user_profile.get("face_shape")
    texture = user_profile.get("texture")
    density = user_profile.get("density")

    face_score = safe_get(rule, "face_shape", {}).get(face_shape, 0)
    texture_score = safe_get(rule, "texture", {}).get(texture, 0)
    density_score = safe_get(rule, "density", {}).get(density, 0)

    hairloss_score, hairloss_reason = hairloss_score_from_profile(rule, user_profile)

    compatibility_score = (
        WEIGHTS["face"] * float(face_score) +
        WEIGHTS["texture"] * float(texture_score) +
        WEIGHTS["density"] * float(density_score) +
        WEIGHTS["hair_loss"] * float(hairloss_score)
    )

    maintenance_penalty = float(rule.get("maintenance", 0) or 0)
    compatibility_score *= (1 - (maintenance_penalty * 0.2))

    if face_score >= 0.85:
        improvement_bonus = ENHANCEMENT_WEIGHT
    elif face_score >= 0.75:
        improvement_bonus = ENHANCEMENT_WEIGHT / 2
    else:
        improvement_bonus = 0.0

    final_score = float(compatibility_score) + float(improvement_bonus)

    breakdown = {
        "face_shape": {"value": face_shape, "rule_score": float(face_score), "weight": WEIGHTS["face"]},
        "texture": {"value": texture, "rule_score": float(texture_score), "weight": WEIGHTS["texture"]},
        "density": {"value": density, "rule_score": float(density_score), "weight": WEIGHTS["density"]},
        "hair_loss": {"rule_score": float(hairloss_score), "weight": WEIGHTS["hair_loss"], "reason": hairloss_reason},
        "maintenance_penalty": maintenance_penalty,
        "improvement_bonus": float(improvement_bonus),
    }

    return round(final_score, 4), breakdown


def build_explanation(style: Dict[str, Any], rule: Dict[str, Any], breakdown: Dict[str, Any]) -> str:
    fx = rule.get("effects", [])
    fx_txt = ", ".join(fx) if isinstance(fx, list) else str(fx)

    parts = [
        f"face_shape={breakdown['face_shape']['value']}({breakdown['face_shape']['rule_score']})",
        f"texture={breakdown['texture']['value']}({breakdown['texture']['rule_score']})",
        f"density={breakdown['density']['value']}({breakdown['density']['rule_score']})",
        f"hair_loss={breakdown['hair_loss']['reason']}({breakdown['hair_loss']['rule_score']})",
    ]
    if breakdown["maintenance_penalty"] > 0:
        parts.append(f"maintenance_penalty={breakdown['maintenance_penalty']}")
    if breakdown["improvement_bonus"] > 0:
        parts.append(f"bonus={breakdown['improvement_bonus']}")
    if fx_txt:
        parts.append(f"effects={fx_txt}")
    return " | ".join(parts)


def diversify(ranked: List[Dict[str, Any]], max_per_category: int = MAX_PER_CATEGORY, top_n: int = 5) -> List[Dict[str, Any]]:
    final: List[Dict[str, Any]] = []
    counts: Dict[str, int] = {}

    for item in ranked:
        base = item.get("base_category", "unknown")
        counts.setdefault(base, 0)
        if counts[base] < max_per_category:
            final.append(item)
            counts[base] += 1
        if len(final) >= top_n:
            break

    return final


def recommend(user_profile: Dict[str, Any], top_n: int = 5, kb_dir: Optional[str] = None) -> List[Dict[str, Any]]:
    validate_weights(WEIGHTS)
    taxonomy, rules = load_taxonomy_and_rules(kb_dir)

    rule_map = {r.get("base_category"): r for r in rules if isinstance(r, dict) and r.get("base_category")}
    if not rule_map:
        raise ValueError("No valid rules found (missing base_category).")

    scored: List[Dict[str, Any]] = []
    for style in taxonomy:
        if not isinstance(style, dict):
            continue
        base = style.get("base_category")
        if not base or base not in rule_map:
            continue

        rule = rule_map[base]
        score, breakdown = compute_score(style, rule, user_profile)
        explanation = build_explanation(style, rule, breakdown)

        scored.append({
            "hairstyle_id": style.get("hairstyle_id"),
            "display_name": style.get("display_name"),
            "base_category": base,
            "score": score,
            "explanation": explanation,
            "breakdown": breakdown,
            "effects": rule.get("effects", [])
        })

    if not scored:
        raise ValueError("No styles scored (taxonomy/rules base_category mismatch or empty KB).")

    ranked = sorted(scored, key=lambda x: x["score"], reverse=True)
    return diversify(ranked, max_per_category=MAX_PER_CATEGORY, top_n=top_n)


def _require_enum(name: str, value: Any, allowed: set):
    if value is None:
        raise ValueError(f"Missing required field '{name}'. Run finalize_profile_meta(meta_path) in your notebook before recommending.")
    v = str(value).lower().strip()
    if v not in allowed:
        raise ValueError(f"Invalid '{name}'='{value}'. Allowed: {sorted(list(allowed))}")
    return v


def recommend_from_meta(meta_json_path: str, top_n: int = 5, kb_dir: Optional[str] = None) -> Dict[str, Any]:
    meta_p = Path(meta_json_path).expanduser().resolve()
    if not meta_p.exists():
        raise FileNotFoundError(f"meta.json not found: {meta_p}")

    meta = _load_json(meta_p)
    if not isinstance(meta, dict):
        raise ValueError("meta.json must be an object/dict.")

    face_shape = meta.get("face_shape") or meta.get("face_shape_label") or meta.get("faceShape")
    texture = meta.get("texture") or meta.get("hair_texture") or meta.get("hair_texture_label") or meta.get("hairTexture")
    density = meta.get("density") or meta.get("hair_density") or meta.get("hair_density_label") or meta.get("hairDensity")

    face_shape = _require_enum("face_shape", face_shape, ALLOWED_FACE)
    texture = _require_enum("texture", texture, ALLOWED_TEXTURE)
    density = _require_enum("density", density, ALLOWED_DENSITY)

    user_profile = {
        "face_shape": face_shape,
        "texture": texture,
        "density": density,
        "baldness_pattern_label": meta.get("baldness_pattern_label") or meta.get("baldness_pattern") or "none",
        "norwood_stage": meta.get("norwood_stage") or meta.get("norwood") or meta.get("norwoodStage"),
    }

    results = recommend(user_profile, top_n=top_n, kb_dir=kb_dir)

    counts: Dict[str, int] = {}
    for r in results:
        counts[r["base_category"]] = counts.get(r["base_category"], 0) + 1

    return {
        "top_5_hairstyle_ids": [r["hairstyle_id"] for r in results[:top_n]],
        "results": results[:top_n],
        "diversification_summary": counts,
        "user_profile_used": user_profile
    }

Overwriting /content/drive/MyDrive/AI_Trichologist/knowledgebase/scripts/reasoning_engine.py


In [49]:
import json
from pathlib import Path
import cv2
import numpy as np

PROFILES_DIR = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(PROFILES_DIR.glob("*_meta.json"))
print("meta files:", len(meta_files))

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def save_json(p: Path, obj):
    p.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def density_from_zone_scores(zs: dict):
    if not isinstance(zs, dict) or not zs:
        return None, None
    vals = [float(v) for v in zs.values() if isinstance(v, (int, float))]
    if not vals:
        return None, None
    scalp = float(np.mean(vals))
    if scalp < 0.12:
        return "high", scalp
    elif scalp < 0.24:
        return "medium", scalp
    else:
        return "low", scalp

def texture_from_image(processed_path: str, mask_path: str):
    if not processed_path or not mask_path:
        return None, None
    ip = Path(processed_path)
    mp = Path(mask_path)
    if not ip.exists() or not mp.exists():
        return None, None

    img = cv2.imread(str(ip))
    msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hair = (msk > 127)
    if hair.sum() < 2000:
        return None, None

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    ang = np.arctan2(gy, gx)

    mag_h = mag[hair]
    ang_h = ang[hair]

    thr = np.percentile(mag_h, 75)
    keep = mag_h > thr
    if keep.sum() < 800:
        return None, None

    ang_k = ang_h[keep]
    R = np.sqrt(np.mean(np.cos(ang_k))**2 + np.mean(np.sin(ang_k))**2)
    circ_var = float(1 - R)

    if circ_var < 0.25:
        return "straight", circ_var
    elif circ_var < 0.45:
        return "wavy", circ_var
    else:
        return "curly", circ_var

def face_shape_from_geometry(geometry_path: str):
    if not geometry_path:
        return None, None
    gp = Path(geometry_path)
    if not gp.exists():
        return None, None
    g = load_json(gp)
    if not isinstance(g, dict):
        return None, None

    for k in ["face_shape", "face_shape_label", "faceShape"]:
        if g.get(k):
            return str(g[k]).lower(), {"source": f"geometry:{k}"}

    fh = g.get("face_height") or g.get("face_length") or g.get("height")
    jw = g.get("jaw_width") or g.get("jawline_width") or g.get("jaw")
    fw = g.get("forehead_width") or g.get("forehead") or g.get("temple_width")

    if all(isinstance(x, (int, float)) and x > 0 for x in [fh, jw]):
        r = float(jw) / float(fh)
        if r < 0.52:
            return "oblong", {"ratio_jaw_to_height": r}
        elif r > 0.62:
            if isinstance(fw, (int, float)) and fw > 0:
                rf = float(fw) / float(jw)
                return ("square" if rf > 0.92 else "round"), {"ratio_jaw_to_height": r, "ratio_forehead_to_jaw": rf}
            return "round", {"ratio_jaw_to_height": r}
        else:
            return "oval", {"ratio_jaw_to_height": r}

    return None, None

updated = 0
for mf in meta_files:
    meta = load_json(mf)

    if meta.get("face_shape") and meta.get("texture") and meta.get("density"):
        continue

    face_shape, face_dbg = face_shape_from_geometry(meta.get("geometry_path"))
    texture, tex_dbg = texture_from_image(meta.get("processed_path"), meta.get("mask_path"))
    density, dens_dbg = density_from_zone_scores(meta.get("baldness_zone_scores"))

    meta.setdefault("profile_source", {})

    if not meta.get("face_shape"):
        meta["face_shape"] = face_shape or "oval"
        meta["profile_source"]["face_shape"] = face_dbg or {"source": "heuristic_default"}

    if not meta.get("texture"):
        meta["texture"] = texture or "straight"
        meta["profile_source"]["texture"] = {"circ_var": tex_dbg} if tex_dbg is not None else {"source": "heuristic_default"}

    if not meta.get("density"):
        meta["density"] = density or "medium"
        meta["profile_source"]["density"] = {"avg_scalp_ratio": dens_dbg} if dens_dbg is not None else {"source": "heuristic_default"}

    meta["profile_version"] = "heuristic_v1"
    save_json(mf, meta)
    updated += 1

print("✅ metas updated:", updated)
print("Sample:", meta_files[0].name, load_json(meta_files[0]).get("face_shape"), load_json(meta_files[0]).get("texture"), load_json(meta_files[0]).get("density"))

meta files: 13
✅ metas updated: 13
Sample: user_20260222_034423_meta.json oval straight medium


In [42]:
from pathlib import Path

profiles_dir = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(profiles_dir.glob("*_meta.json"))

print("Meta count:", len(meta_files))
print("Example:", meta_files[0].name)

meta_path = str(meta_files[0])
meta_path

Meta count: 13
Example: user_20260222_034423_meta.json


'/content/drive/MyDrive/AI_Trichologist/data/profiles/user_20260222_034423_meta.json'

In [43]:
import json

meta = json.loads(Path(meta_path).read_text(encoding="utf-8"))
print("Keys:", list(meta.keys())[:25], "...")
print("baldness_pattern_label:", meta.get("baldness_pattern_label"))
print("baldness_zone_scores:", meta.get("baldness_zone_scores"))

Keys: ['image_path', 'original_shape', 'face_detected', 'face_box_xyxy', 'face_area_ratio', 'clahe', 'color_space', 'normalize', 'target_size', 'post_crop_shape', 'resized_shape', 'dtype', 'min', 'max', 'mean', 'raw_path', 'processed_path', 'image_id', 'baldness_path', 'zones_overlay_path', 'bald_model', 'bald_summary', 'geometry_path', 'landmark_model', 'landmark_count'] ...
baldness_pattern_label: diffuse
baldness_zone_scores: {'frontal': 0.06618133686300463, 'mid': 0.29283815354869897, 'crown': 0.2733668341708543}


In [56]:
import json
from pathlib import Path
from collections import Counter

rules_path = Path("/content/drive/MyDrive/AI_Trichologist/knowledgebase/data_knowledge/hairstyle_rules.json")
rules = json.loads(rules_path.read_text(encoding="utf-8"))

cnt = 0
zeros = 0
missing = 0
examples = []

for r in rules:
    hl = r.get("hair_loss", {})
    if not isinstance(hl, dict):
        missing += 1
        continue
    fr = hl.get("receding_hairline", None)
    cr = hl.get("crown_thinning", None)
    if fr is None or cr is None:
        missing += 1
        continue
    cnt += 1
    if float(fr) == 0.0 and float(cr) == 0.0:
        zeros += 1
        if len(examples) < 5:
            examples.append((r.get("base_category"), fr, cr))

print("rules with both keys:", cnt)
print("rules missing keys:", missing)
print("rules both zero:", zeros)
print("examples (base, fr, cr):", examples)

rules with both keys: 21
rules missing keys: 0
rules both zero: 14
examples (base, fr, cr): [('crew', 0, 0), ('fade', 0, 0), ('undercut', 0, 0), ('quiff', 0, 0), ('pompadour', 0, 0)]


Step A — Inspect what’s inside

In [62]:
import json
from pathlib import Path

gp = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles/user_20260222_034423_geometry.json")
g = json.loads(gp.read_text(encoding="utf-8"))

geo = g.get("geometry", {})
print("type(geo):", type(geo))
print("geo keys:", list(geo.keys())[:80])

# If there are nested dicts, show them
for k,v in geo.items():
    if isinstance(v, dict):
        print("\nNested dict key:", k, "->", list(v.keys())[:40])

type(geo): <class 'dict'>
geo keys: ['distances_px', 'ratios', 'landmark_indices_used', 'image_size', 'symmetry']

Nested dict key: distances_px -> ['face_width', 'face_height', 'jaw_width', 'cheekbone_width', 'forehead_width']

Nested dict key: ratios -> ['width_height', 'jaw_cheekbone', 'forehead_cheekbone']

Nested dict key: landmark_indices_used -> ['left_cheek', 'right_cheek', 'top', 'chin', 'jaw_left', 'jaw_right', 'cheekbone_left', 'cheekbone_right', 'forehead_left', 'forehead_right', 'midline']

Nested dict key: image_size -> ['width', 'height']

Nested dict key: symmetry -> ['enabled', 'midline_index', 'pair_count', 'mean_mirror_error_px', 'normalized_error', 'symmetry_score', 'pairs']


In [63]:
import json
from pathlib import Path
from collections import Counter

profiles_dir = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(profiles_dir.glob("*_meta.json"))

textures = []
for p in meta_files:
    m = json.loads(p.read_text(encoding="utf-8"))
    textures.append(m.get("texture"))

print("texture dist:", Counter(textures))

texture dist: Counter({'straight': 13})


In [64]:
import json
from pathlib import Path
import cv2
import numpy as np
from collections import Counter

PROFILES_DIR = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(PROFILES_DIR.glob("*_meta.json"))

def texture_from_image(processed_path: str, mask_path: str):
    ip = Path(processed_path) if processed_path else None
    mp = Path(mask_path) if mask_path else None
    if not ip or not mp or (not ip.exists()) or (not mp.exists()):
        return None

    img = cv2.imread(str(ip))
    msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hair = (msk > 127)
    if int(hair.sum()) < 2000:
        return None

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)[hair]
    ang = np.arctan2(gy, gx)[hair]

    thr = np.percentile(mag, 70)   # relaxed from 75 to 70
    keep = mag > thr
    if int(keep.sum()) < 500:      # relaxed from 800 to 500
        return None

    ang_k = ang[keep]
    R = np.sqrt(np.mean(np.cos(ang_k))**2 + np.mean(np.sin(ang_k))**2)
    circ_var = float(1 - R)

    if circ_var < 0.22:
        return "straight"
    elif circ_var < 0.42:
        return "wavy"
    else:
        return "curly"

updated = 0
for mf in meta_files:
    meta = json.loads(mf.read_text(encoding="utf-8"))
    tex = texture_from_image(meta.get("processed_path"), meta.get("mask_path"))
    if tex:
        meta["texture"] = tex
        meta.setdefault("profile_source", {})
        meta["profile_source"]["texture"] = {"source": "heuristic_edges_v2"}
        mf.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
        updated += 1

print("✅ texture updated:", updated)

# show new distribution
textures = []
for mf in meta_files:
    m = json.loads(mf.read_text(encoding="utf-8"))
    textures.append(m.get("texture"))
print("texture dist:", Counter(textures))

✅ texture updated: 13
texture dist: Counter({'curly': 13})


In [65]:
import json
from pathlib import Path

rules_path = Path("/content/drive/MyDrive/AI_Trichologist/knowledgebase/data_knowledge/hairstyle_rules.json")
rules = json.loads(rules_path.read_text(encoding="utf-8"))

zeros = []
for r in rules:
    hl = r.get("hair_loss", {})
    fr = float(hl.get("receding_hairline", 0))
    cr = float(hl.get("crown_thinning", 0))
    if fr == 0.0 and cr == 0.0:
        zeros.append(r.get("base_category"))

print("both-zero categories:", zeros)
print("count:", len(zeros))

both-zero categories: []
count: 0


In [66]:
import json
from pathlib import Path
from collections import Counter

PROFILES_DIR = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(PROFILES_DIR.glob("*_meta.json"))

def face_shape_from_geo(geo: dict):
    r = geo.get("ratios", {}) if isinstance(geo, dict) else {}
    d = geo.get("distances_px", {}) if isinstance(geo, dict) else {}

    width_height = r.get("width_height")  # face_width / face_height
    jaw_cheek = r.get("jaw_cheekbone")    # jaw_width / cheekbone_width
    fore_cheek = r.get("forehead_cheekbone")  # forehead_width / cheekbone_width

    if not all(isinstance(x, (int, float)) for x in [width_height, jaw_cheek, fore_cheek]):
        return None, None

    wh = float(width_height)
    jc = float(jaw_cheek)
    fc = float(fore_cheek)

    # Simple, defensible heuristic rules
    # - Oblong: long face (narrow relative width)
    if wh < 0.72:
        return "oblong", {"width_height": wh, "jaw_cheekbone": jc, "forehead_cheekbone": fc}

    # - Square: jaw ~ cheekbone and forehead ~ cheekbone (strong angles)
    if jc > 0.92 and fc > 0.90:
        return "square", {"width_height": wh, "jaw_cheekbone": jc, "forehead_cheekbone": fc}

    # - Round: wide face, softer jaw (jaw smaller than cheekbone)
    if wh > 0.82 and jc < 0.88:
        return "round", {"width_height": wh, "jaw_cheekbone": jc, "forehead_cheekbone": fc}

    # - Heart: forehead wider than jaw
    if fc > 0.94 and jc < 0.86:
        return "heart", {"width_height": wh, "jaw_cheekbone": jc, "forehead_cheekbone": fc}

    # Default: oval
    return "oval", {"width_height": wh, "jaw_cheekbone": jc, "forehead_cheekbone": fc}

updated = 0
faces = []

for mf in meta_files:
    meta = json.loads(mf.read_text(encoding="utf-8"))
    gp = meta.get("geometry_path")
    if not gp or not Path(gp).exists():
        faces.append(meta.get("face_shape"))
        continue

    g = json.loads(Path(gp).read_text(encoding="utf-8"))
    geo = g.get("geometry", {})
    fs, dbg = face_shape_from_geo(geo)

    if fs:
        meta["face_shape"] = fs
        meta.setdefault("profile_source", {})
        meta["profile_source"]["face_shape"] = {"source": "geometry.ratios", **dbg}
        mf.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
        updated += 1

    faces.append(meta.get("face_shape"))

print("✅ face_shape updated:", updated)
print("face_shape dist:", Counter(faces))

✅ face_shape updated: 13
face_shape dist: Counter({'heart': 10, 'round': 3})


In [68]:
import json
from pathlib import Path
import cv2
import numpy as np
from collections import Counter

PROFILES_DIR = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(PROFILES_DIR.glob("*_meta.json"))

def texture_v3(processed_path: str, mask_path: str):
    ip = Path(processed_path) if processed_path else None
    mp = Path(mask_path) if mask_path else None
    if not ip or not mp or (not ip.exists()) or (not mp.exists()):
        return None, None

    img = cv2.imread(str(ip))
    msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hair = (msk > 127)
    hair_n = int(hair.sum())
    if hair_n < 3000:
        return None, None

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    ang = np.arctan2(gy, gx)

    mag_h = mag[hair]
    ang_h = ang[hair]

    # strong edges threshold uses 80th percentile for stability
    thr = np.percentile(mag_h, 80)
    strong = mag_h > thr
    strong_n = int(strong.sum())
    if strong_n < 600:
        return "straight", {"reason": "low_edges", "hair_n": hair_n, "strong_n": strong_n}

    # edge density
    edge_density = strong_n / hair_n

    ang_k = ang_h[strong]
    R = np.sqrt(np.mean(np.cos(ang_k))**2 + np.mean(np.sin(ang_k))**2)
    circ_var = float(1 - R)

    # Classification rules
    # Straight if low directional variance OR low edge density
    if circ_var < 0.28 or edge_density < 0.10:
        tex = "straight"
    # Curly only when both are high
    elif circ_var > 0.48 and edge_density > 0.18:
        tex = "curly"
    else:
        tex = "wavy"

    return tex, {"circ_var": circ_var, "edge_density": edge_density, "hair_n": hair_n, "strong_n": strong_n}

updated = 0
textures = []
for mf in meta_files:
    meta = json.loads(mf.read_text(encoding="utf-8"))
    tex, dbg = texture_v3(meta.get("processed_path"), meta.get("mask_path"))
    if tex:
        meta["texture"] = tex
        meta.setdefault("profile_source", {})
        meta["profile_source"]["texture"] = {"source": "heuristic_texture_v3", **dbg}
        mf.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
        updated += 1
    textures.append(meta.get("texture"))

print("✅ texture updated:", updated)
print("texture dist:", Counter(textures))

✅ texture updated: 13
texture dist: Counter({'curly': 13})


In [73]:
import json
from pathlib import Path
import cv2
import numpy as np
from collections import Counter

PROFILES_DIR = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(PROFILES_DIR.glob("*_meta.json"))

def texture_v4(processed_path: str, mask_path: str):
    ip = Path(processed_path) if processed_path else None
    mp = Path(mask_path) if mask_path else None
    if not ip or not mp or (not ip.exists()) or (not mp.exists()):
        return None, None

    img = cv2.imread(str(ip))
    msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hair = (msk > 127).astype(np.uint8)

    # remove boundary edges (big cause of fake "curly")
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    hair_core = cv2.erode(hair, k, iterations=2).astype(bool)

    hair_n = int(hair_core.sum())
    if hair_n < 2000:
        return None, None

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    ang = np.arctan2(gy, gx)

    mag_h = mag[hair_core]
    ang_h = ang[hair_core]

    # ABSOLUTE threshold based on data scale (your mean ~69, p75 ~89, p90 ~155)
    # We'll use a fixed cutoff so edge_density varies across images.
    thr = 90.0
    strong = mag_h > thr
    strong_n = int(strong.sum())
    edge_density = strong_n / hair_n

    # if not enough strong edges, treat as straight-ish
    if strong_n < 500:
        return "straight", {"circ_var": None, "edge_density": edge_density, "hair_n": hair_n, "strong_n": strong_n, "thr": thr}

    ang_k = ang_h[strong]
    R = np.sqrt(np.mean(np.cos(ang_k))**2 + np.mean(np.sin(ang_k))**2)
    circ_var = float(1 - R)

    # Classify with both signals
    if circ_var < 0.30:
        tex = "straight"
    elif circ_var < 0.52:
        tex = "wavy"
    else:
        # only call curly if there is substantial edge structure
        tex = "curly" if edge_density > 0.08 else "wavy"

    return tex, {"circ_var": circ_var, "edge_density": edge_density, "hair_n": hair_n, "strong_n": strong_n, "thr": thr}

rows = []
updated = 0
for mf in meta_files:
    meta = json.loads(mf.read_text(encoding="utf-8"))
    tex, dbg = texture_v4(meta.get("processed_path"), meta.get("mask_path"))
    rows.append((mf.name, tex, None if not dbg else dbg.get("circ_var"), None if not dbg else dbg.get("edge_density")))

    if tex:
        meta["texture"] = tex
        meta.setdefault("profile_source", {})
        meta["profile_source"]["texture"] = {"source": "heuristic_texture_v4", **(dbg or {})}
        mf.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
        updated += 1

print("✅ texture updated:", updated)
print("texture dist:", Counter([r[1] for r in rows]))

print("\nname | tex | circ_var | edge_density")
for name, tex, cv, ed in rows:
    print(name, "|", tex, "|", (None if cv is None else round(float(cv), 3)), "|", (None if ed is None else round(float(ed), 3)))

✅ texture updated: 13
texture dist: Counter({'curly': 13})

name | tex | circ_var | edge_density
user_20260222_034423_meta.json | curly | 0.953 | 0.243
user_20260222_050806_meta.json | curly | 0.909 | 0.233
user_20260222_151340_544_meta.json | curly | 0.939 | 0.502
user_20260222_153307_967_meta.json | curly | 0.949 | 0.367
user_20260222_153328_141_meta.json | curly | 0.936 | 0.219
user_20260222_153353_899_meta.json | curly | 0.973 | 0.437
user_20260222_153414_647_meta.json | curly | 0.932 | 0.128
user_20260222_153434_876_meta.json | curly | 0.952 | 0.437
user_20260222_153452_139_meta.json | curly | 0.971 | 0.591
user_20260222_153736_798_meta.json | curly | 0.946 | 0.416
user_20260223_075002_778_meta.json | curly | 0.84 | 0.213
user_20260223_075018_627_meta.json | curly | 0.832 | 0.094
user_20260223_075109_087_meta.json | curly | 0.966 | 0.088


In [82]:
import json
from pathlib import Path
import cv2
import numpy as np
from collections import Counter

PROFILES_DIR = Path("/content/drive/MyDrive/AI_Trichologist/data/profiles")
meta_files = sorted(PROFILES_DIR.glob("*_meta.json"))

def texture_score(processed_path: str, mask_path: str):
    ip = Path(processed_path) if processed_path else None
    mp = Path(mask_path) if mask_path else None
    if not ip or not mp or (not ip.exists()) or (not mp.exists()):
        return None

    img = cv2.imread(str(ip))
    msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hair = (msk > 127).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    hair_core = cv2.erode(hair, k, iterations=2).astype(bool)
    if int(hair_core.sum()) < 2000:
        return None

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)

    mag_h = mag[hair_core]
    # Use robust statistic: coefficient of variation of gradient magnitude
    m = float(np.mean(mag_h))
    s = float(np.std(mag_h))
    if m <= 1e-6:
        return None
    return s / m  # higher => more micro-structure (curlier / more textured)

scores = []
paths = []
for mf in meta_files:
    meta = json.loads(mf.read_text(encoding="utf-8"))
    sc = texture_score(meta.get("processed_path"), meta.get("mask_path"))
    scores.append(sc)
    paths.append(mf)

valid = [x for x in scores if x is not None]
if len(valid) < 5:
    raise ValueError("Too few valid texture scores; images/masks not readable.")

q33 = float(np.quantile(valid, 0.33))
q66 = float(np.quantile(valid, 0.66))

def label_from_score(sc):
    if sc is None:
        return "wavy"  # neutral fallback
    if sc < q33:
        return "straight"
    elif sc < q66:
        return "wavy"
    else:
        return "curly"

updated = 0
labels = []
print("texture_score quantiles:", {"q33": round(q33,4), "q66": round(q66,4)})

for mf, sc in zip(paths, scores):
    meta = json.loads(mf.read_text(encoding="utf-8"))
    tex = label_from_score(sc)
    meta["texture"] = tex
    meta.setdefault("profile_source", {})
    meta["profile_source"]["texture"] = {"source": "heuristic_texture_v5_cv", "score_cv": (None if sc is None else float(sc)), "q33": q33, "q66": q66}
    mf.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    updated += 1
    labels.append(tex)

print("✅ texture updated:", updated)
print("texture dist:", Counter(labels))

# show per-file
for mf, sc, tex in zip([p.name for p in paths], scores, labels):
    print(mf, "| score:", (None if sc is None else round(float(sc),4)), "|", tex)

texture_score quantiles: {'q33': 0.7937, 'q66': 0.9736}
✅ texture updated: 13
texture dist: Counter({'curly': 5, 'straight': 4, 'wavy': 4})
user_20260222_034423_meta.json | score: 1.0288 | curly
user_20260222_050806_meta.json | score: 0.7801 | straight
user_20260222_151340_544_meta.json | score: 0.8811 | wavy
user_20260222_153307_967_meta.json | score: 0.9703 | wavy
user_20260222_153328_141_meta.json | score: 1.0779 | curly
user_20260222_153353_899_meta.json | score: 0.88 | wavy
user_20260222_153414_647_meta.json | score: 0.9739 | curly
user_20260222_153434_876_meta.json | score: 0.9918 | curly
user_20260222_153452_139_meta.json | score: 0.7943 | wavy
user_20260222_153736_798_meta.json | score: 0.7669 | straight
user_20260223_075002_778_meta.json | score: 1.0393 | curly
user_20260223_075018_627_meta.json | score: 0.7049 | straight
user_20260223_075109_087_meta.json | score: 0.7473 | straight


In [83]:
import os, sys, importlib
from pathlib import Path

KB_DIR="/content/drive/MyDrive/AI_Trichologist/knowledgebase"
os.environ["KB_DIR"]=KB_DIR
sys.path.insert(0, str(Path(KB_DIR)/"scripts"))

import reasoning_engine as re_engine
importlib.reload(re_engine)

meta_path="/content/drive/MyDrive/AI_Trichologist/data/profiles/user_20260222_034423_meta.json"
out=re_engine.recommend_from_meta(meta_path, top_n=5)

print("user_profile_used:", out["user_profile_used"])
print("top5:", out["top_5_hairstyle_ids"])
print("explain #1:", out["results"][0]["explanation"])

user_profile_used: {'face_shape': 'heart', 'texture': 'curly', 'density': 'medium', 'baldness_pattern_label': 'diffuse', 'norwood_stage': None}
top5: ['textured_fringe', 'side_swept_fringe', 'high_fade', 'bald_fade_with_full_beard', 'classic_edgar']
explain #1: face_shape=heart(0.9) | texture=curly(0.7) | density=medium(0.8) | hair_loss=pattern=diffuse→avg(frontal,crown)(0.45) | maintenance_penalty=0.5 | bonus=0.1 | effects=softening, youthful, stylish, hides forehead


In [85]:
meta_path = "/content/drive/MyDrive/AI_Trichologist/data/profiles/user_20260222_034423_meta.json"
out = re_engine.recommend_from_meta(meta_path)
out["top_5_hairstyle_ids"], out["diversification_summary"]

(['textured_fringe',
  'side_swept_fringe',
  'high_fade',
  'bald_fade_with_full_beard',
  'classic_edgar'],
 {'fringe': 2, 'fade': 2, 'edgar': 1})

In [86]:
meta_path="/content/drive/MyDrive/AI_Trichologist/data/profiles/user_20260222_034423_meta.json"
out = re_engine.recommend_from_meta(meta_path, top_n=5)

for r in out["results"]:
    print(r["hairstyle_id"], "|", r["base_category"], "|", r["score"])
    print("  ", r["explanation"])

textured_fringe | fringe | 0.7863
   face_shape=heart(0.9) | texture=curly(0.7) | density=medium(0.8) | hair_loss=pattern=diffuse→avg(frontal,crown)(0.45) | maintenance_penalty=0.5 | bonus=0.1 | effects=softening, youthful, stylish, hides forehead
side_swept_fringe | fringe | 0.7863
   face_shape=heart(0.9) | texture=curly(0.7) | density=medium(0.8) | hair_loss=pattern=diffuse→avg(frontal,crown)(0.45) | maintenance_penalty=0.5 | bonus=0.1 | effects=softening, youthful, stylish, hides forehead
high_fade | fade | 0.7346
   face_shape=heart(0.8) | texture=curly(0.9) | density=medium(0.9) | hair_loss=pattern=diffuse→avg(frontal,crown)(0.6) | maintenance_penalty=0.8 | bonus=0.05 | effects=sharp, clean, modern, polished, high contrast
bald_fade_with_full_beard | fade | 0.7346
   face_shape=heart(0.8) | texture=curly(0.9) | density=medium(0.9) | hair_loss=pattern=diffuse→avg(frontal,crown)(0.6) | maintenance_penalty=0.8 | bonus=0.05 | effects=sharp, clean, modern, polished, high contrast
clas